# Week 3, Notebook 1: Autoencoder & VAE from Scratch
## Understanding Generative Models at the Lowest Level

**What you'll build:** An autoencoder and a variational autoencoder (VAE) in pure NumPy.

**Curriculum points (all integrated):**
- All 7 principles applied to generative modeling
- Plus: The reparameterization trick for VAEs

**Time estimate:** 45–60 minutes

---
### The Generative Idea
Instead of classifying data, we learn to **reconstruct** it.  
An autoencoder compresses data to a bottleneck (latent space), then reconstructs it.  
A **Variational** Autoencoder makes the latent space smooth → we can SAMPLE new data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# ============================================================
# Dataset: Simple 2D patterns we can visualize
# ============================================================
def make_dataset(n=1000):
    """Generate a mixture of 2D Gaussians (4 clusters)."""
    centers = np.array([[2, 2], [-2, 2], [-2, -2], [2, -2]])
    points = []
    labels = []
    for i, c in enumerate(centers):
        pts = np.random.randn(n // 4, 2) * 0.5 + c
        points.append(pts)
        labels.extend([i] * (n // 4))
    return np.vstack(points).astype(np.float32), np.array(labels)

X_data, y_labels = make_dataset(1000)

plt.figure(figsize=(6, 6))
for i in range(4):
    mask = y_labels == i
    plt.scatter(X_data[mask, 0], X_data[mask, 1], s=10, alpha=0.5, label=f'Cluster {i}')
plt.title('Training Data: 4 Gaussian Clusters')
plt.legend()
plt.grid(True, alpha=0.2)
plt.axis('equal')
plt.savefig('w3_01_data.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 1: Autoencoder — Compress and Reconstruct

In [ ]:
# ============================================================
# Autoencoder: Encoder (2D → 1D bottleneck) → Decoder (1D → 2D)
# ============================================================

class AutoEncoder:
    """Simple autoencoder in pure NumPy."""
    
    def __init__(self, input_dim=2, hidden_dim=8, latent_dim=1):
        # Encoder weights
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, latent_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(latent_dim)
        
        # Decoder weights  
        self.W3 = np.random.randn(latent_dim, hidden_dim) * np.sqrt(2.0 / latent_dim)
        self.b3 = np.zeros(hidden_dim)
        self.W4 = np.random.randn(hidden_dim, input_dim) * np.sqrt(2.0 / hidden_dim)
        self.b4 = np.zeros(input_dim)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_grad(self, x):
        return (x > 0).astype(float)
    
    def encode(self, X):
        """Compress input to latent space."""
        self.h1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.h1)
        self.z = self.a1 @ self.W2 + self.b2  # Latent representation
        return self.z
    
    def decode(self, z):
        """Reconstruct from latent space."""
        self.h3 = z @ self.W3 + self.b3
        self.a3 = self.relu(self.h3)
        self.x_hat = self.a3 @ self.W4 + self.b4  # Reconstruction (linear output)
        return self.x_hat
    
    def forward(self, X):
        self.X = X
        z = self.encode(X)
        x_hat = self.decode(z)
        return x_hat
    
    def backward(self, lr=0.001):
        """Full backprop through encoder + decoder."""
        m = self.X.shape[0]
        
        # Reconstruction loss gradient: d/dx_hat (x - x_hat)^2 = -2(x - x_hat)/m
        d_xhat = -2 * (self.X - self.x_hat) / m
        
        # Decoder gradients
        dW4 = self.a3.T @ d_xhat
        db4 = d_xhat.mean(axis=0)
        d_a3 = d_xhat @ self.W4.T
        d_h3 = d_a3 * self.relu_grad(self.h3)
        dW3 = self.z.T @ d_h3
        db3 = d_h3.mean(axis=0)
        
        # Encoder gradients (chain rule continues!)
        d_z = d_h3 @ self.W3.T
        dW2 = self.a1.T @ d_z
        db2 = d_z.mean(axis=0)
        d_a1 = d_z @ self.W2.T
        d_h1 = d_a1 * self.relu_grad(self.h1)
        dW1 = self.X.T @ d_h1
        db1 = d_h1.mean(axis=0)
        
        # Update all weights
        for W, dW in [(self.W1, dW1), (self.W2, dW2), (self.W3, dW3), (self.W4, dW4)]:
            W -= lr * dW
        for b, db in [(self.b1, db1), (self.b2, db2), (self.b3, db3), (self.b4, db4)]:
            b -= lr * db
    
    def train(self, X, epochs=2000, lr=0.005, batch_size=64):
        losses = []
        for epoch in range(epochs):
            # Mini-batch
            idx = np.random.choice(len(X), batch_size)
            batch = X[idx]
            
            x_hat = self.forward(batch)
            loss = np.mean((batch - x_hat) ** 2)
            losses.append(loss)
            self.backward(lr=lr)
            
            if epoch % 500 == 0:
                print(f"  Epoch {epoch:5d} | Reconstruction Loss: {loss:.6f}")
        return losses

# Train the autoencoder
ae = AutoEncoder(input_dim=2, hidden_dim=16, latent_dim=1)
print("Training autoencoder (2D → 1D → 2D)...")
losses = ae.train(X_data, epochs=3000, lr=0.005)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(losses)
axes[0].set_title('Reconstruction Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.2)

x_hat = ae.forward(X_data)
axes[1].scatter(X_data[:, 0], X_data[:, 1], c='steelblue', s=10, alpha=0.3, label='Original')
axes[1].scatter(x_hat[:, 0], x_hat[:, 1], c='coral', s=10, alpha=0.3, label='Reconstructed')
axes[1].set_title('Original vs Reconstructed')
axes[1].legend()
axes[1].grid(True, alpha=0.2)
axes[1].set_aspect('equal')

z_vals = ae.encode(X_data)
axes[2].scatter(z_vals, np.zeros_like(z_vals), c=y_labels, s=10, cmap='tab10', alpha=0.5)
axes[2].set_title('Latent Space (1D)')
axes[2].set_xlabel('z')

plt.suptitle('AUTOENCODER: Compress 2D → 1D → Reconstruct', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w3_01_autoencoder.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: Variational Autoencoder (VAE)

The key upgrade: instead of encoding to a POINT, encode to a DISTRIBUTION.  
This makes the latent space smooth → we can sample new data!

**The Reparameterization Trick:**  
Instead of sampling z ~ N(μ, σ²), compute z = μ + σ · ε where ε ~ N(0,1).  
This makes the sampling differentiable → backprop works!

In [ ]:
# ============================================================
# Variational Autoencoder (VAE) — the reparameterization trick
# ============================================================

class VAE:
    """Variational Autoencoder in pure NumPy."""
    
    def __init__(self, input_dim=2, hidden_dim=16, latent_dim=2):
        s1 = np.sqrt(2.0 / input_dim)
        s2 = np.sqrt(2.0 / hidden_dim)
        s3 = np.sqrt(2.0 / latent_dim)
        
        # Encoder → outputs mean AND log-variance
        self.W1 = np.random.randn(input_dim, hidden_dim) * s1
        self.b1 = np.zeros(hidden_dim)
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * s2
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * s2
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder
        self.W3 = np.random.randn(latent_dim, hidden_dim) * s3
        self.b3 = np.zeros(hidden_dim)
        self.W4 = np.random.randn(hidden_dim, input_dim) * s2
        self.b4 = np.zeros(input_dim)
    
    def relu(self, x): return np.maximum(0, x)
    def relu_grad(self, x): return (x > 0).astype(float)
    
    def encode(self, X):
        self.h1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.h1)
        self.mu = self.a1 @ self.W_mu + self.b_mu
        self.logvar = self.a1 @ self.W_logvar + self.b_logvar
        return self.mu, self.logvar
    
    def reparameterize(self, mu, logvar):
        """The trick: z = mu + std * epsilon, where epsilon ~ N(0,1)."""
        self.std = np.exp(0.5 * logvar)
        self.eps = np.random.randn(*mu.shape)
        z = mu + self.std * self.eps
        return z
    
    def decode(self, z):
        self.h3 = z @ self.W3 + self.b3
        self.a3 = self.relu(self.h3)
        self.x_hat = self.a3 @ self.W4 + self.b4
        return self.x_hat
    
    def forward(self, X):
        self.X = X
        mu, logvar = self.encode(X)
        self.z = self.reparameterize(mu, logvar)
        x_hat = self.decode(self.z)
        return x_hat, mu, logvar
    
    def loss(self, X, x_hat, mu, logvar):
        """VAE Loss = Reconstruction + KL Divergence."""
        recon = np.mean((X - x_hat) ** 2)
        kl = -0.5 * np.mean(1 + logvar - mu**2 - np.exp(logvar))
        return recon + 0.1 * kl, recon, kl
    
    def backward(self, lr=0.001):
        m = self.X.shape[0]
        
        # dL/dx_hat (reconstruction gradient)
        d_xhat = -2 * (self.X - self.x_hat) / m
        
        # Decoder grads
        dW4 = self.a3.T @ d_xhat
        db4 = d_xhat.mean(axis=0)
        d_a3 = d_xhat @ self.W4.T
        d_h3 = d_a3 * self.relu_grad(self.h3)
        dW3 = self.z.T @ d_h3
        db3 = d_h3.mean(axis=0)
        d_z = d_h3 @ self.W3.T
        
        # Through reparameterization: dz/dmu = 1, dz/dlogvar = 0.5*std*eps
        d_mu = d_z + 0.1 * (2 * self.mu / m)  # + KL gradient
        d_logvar = d_z * 0.5 * self.std * self.eps + 0.1 * (-0.5 * (1 - np.exp(self.logvar)) / m)
        
        # Encoder grads
        dW_mu = self.a1.T @ d_mu
        db_mu = d_mu.mean(axis=0)
        dW_logvar = self.a1.T @ d_logvar
        db_logvar = d_logvar.mean(axis=0)
        
        d_a1 = d_mu @ self.W_mu.T + d_logvar @ self.W_logvar.T
        d_h1 = d_a1 * self.relu_grad(self.h1)
        dW1 = self.X.T @ d_h1
        db1 = d_h1.mean(axis=0)
        
        # Clip gradients to prevent explosions
        for dW in [dW1, dW_mu, dW_logvar, dW3, dW4]:
            np.clip(dW, -1.0, 1.0, out=dW)
        
        # Update
        for W, dW in [(self.W1, dW1), (self.W_mu, dW_mu), (self.W_logvar, dW_logvar),
                       (self.W3, dW3), (self.W4, dW4)]:
            W -= lr * dW
        for b, db in [(self.b1, db1), (self.b_mu, db_mu), (self.b_logvar, db_logvar),
                       (self.b3, db3), (self.b4, db4)]:
            b -= lr * db
    
    def train_loop(self, X, epochs=3000, lr=0.003, batch_size=64):
        history = {'total': [], 'recon': [], 'kl': []}
        for epoch in range(epochs):
            idx = np.random.choice(len(X), batch_size)
            batch = X[idx]
            
            x_hat, mu, logvar = self.forward(batch)
            total, recon, kl = self.loss(batch, x_hat, mu, logvar)
            history['total'].append(total)
            history['recon'].append(recon)
            history['kl'].append(kl)
            
            self.backward(lr=lr)
            
            if epoch % 1000 == 0:
                print(f"  Epoch {epoch:5d} | Total: {total:.4f} | "
                      f"Recon: {recon:.4f} | KL: {kl:.4f}")
        return history


# Train VAE with 2D latent space
vae = VAE(input_dim=2, hidden_dim=32, latent_dim=2)
print("Training VAE (2D → 2D latent → 2D)...")
history = vae.train_loop(X_data, epochs=5000, lr=0.003)

In [ ]:
# ============================================================
# Visualize VAE results
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Loss curves
axes[0, 0].plot(history['recon'], label='Reconstruction', alpha=0.7)
axes[0, 0].plot(history['kl'], label='KL Divergence', alpha=0.7)
axes[0, 0].set_title('VAE Loss Components')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.2)

# Latent space
x_hat, mu, logvar = vae.forward(X_data)
z_encoded = vae.z
axes[0, 1].scatter(z_encoded[:, 0], z_encoded[:, 1], c=y_labels, s=10, cmap='tab10', alpha=0.5)
axes[0, 1].set_title('Latent Space (colored by cluster)')
axes[0, 1].set_xlabel('z₁')
axes[0, 1].set_ylabel('z₂')
axes[0, 1].grid(True, alpha=0.2)

# Reconstruction
axes[1, 0].scatter(X_data[:, 0], X_data[:, 1], c='steelblue', s=10, alpha=0.2, label='Original')
axes[1, 0].scatter(x_hat[:, 0], x_hat[:, 1], c='coral', s=10, alpha=0.2, label='Reconstructed')
axes[1, 0].set_title('Original vs Reconstructed')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.2)
axes[1, 0].set_aspect('equal')

# GENERATION: Sample from latent space and decode
z_samples = np.random.randn(500, 2)  # Sample from N(0,1)
generated = vae.decode(z_samples)
axes[1, 1].scatter(X_data[:, 0], X_data[:, 1], c='steelblue', s=10, alpha=0.1, label='Real data')
axes[1, 1].scatter(generated[:, 0], generated[:, 1], c='red', s=10, alpha=0.3, label='GENERATED')
axes[1, 1].set_title('🎨 GENERATED Samples (from random z)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.2)
axes[1, 1].set_aspect('equal')

plt.suptitle('VARIATIONAL AUTOENCODER — Generation from Scratch!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w3_01_vae.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n🎨 GENERATED data is in the bottom-right plot!")
print("   We sampled random z from N(0,1) and decoded → NEW data points!")
print("   This is the core idea behind generative AI.")

## ✅ Self-Check

- [ ] You can explain: Autoencoder vs VAE → VAE adds distribution + KL divergence
- [ ] You understand the reparameterization trick: z = μ + σ·ε makes sampling differentiable
- [ ] You GENERATED new data points from your VAE
- [ ] You can explain why KL divergence pushes the latent space toward N(0,1)

## ➡️ Next: `W3_02_PyTorch_Generative_AI.ipynb` — Full PyTorch VAE on images

## Visualizing the Autoencoder Architecture

The classic 'hourglass' shape mapping high-dimensional data down to a Latent Bottleneck, and reconstructing it back.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
layers = [
    ('Input (High Dim)', 5),
    ('Encoder Hidden', 3), 
    ('Latent Bottleneck', 1), 
    ('Decoder Hidden', 3), 
    ('Reconstruction', 5)
]
pos = {}
idx = 0
for i, (name, count) in enumerate(layers):
    for j in range(count):
        node = f"{name}_{j}"
        pos[node] = (i * 2, count / 2.0 - j)
        G.add_node(node)
        
        # connect to previous layer
        if i > 0:
            prev_name, prev_count = layers[i-1]
            for k in range(prev_count):
                G.add_edge(f"{prev_name}_{k}", node)

plt.figure(figsize=(10, 5))
nx.draw(G, pos, node_color='lightblue', node_size=800, arrowsize=10)
plt.title("Autoencoder Hourglass Architecture")

plt.savefig('w3_01_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
